# V3 — Hyperparameter search de Flan-T5-base

Tres configuraciones de full fine-tuning exploradas para cerrar el gap de ~2
puntos respecto al techo BART/Pegasus observado en V2.

| Config   | train_subset | epochs | learning_rate | Hipótesis                          |
|----------|--------------|--------|---------------|------------------------------------|
| v3_t5_A  | 10.000       | 2      | 3e-5          | Baseline (= V2, sanity check)      |
| v3_t5_B  | 10.000       | 3      | 1e-4          | Higher LR + more epochs            |
| v3_t5_C  | 20.000       | 2      | 3e-5          | More data, same regime             |

El resto de parámetros se mantiene constante:
- fp32 + AdamW (por estabilidad numérica de T5)
- Batch efectivo 16 (per_device=8 × grad_accum=2)
- `weight_decay=0.01`, `warmup_ratio=0.1`

**Criterio de selección:** mejor ROUGE-L sobre el test subset de 200 muestras
(el mismo que V1 y V2, para comparabilidad directa).

**Tiempo estimado:** ~3-4h en total (A y B ~1h cada uno, C ~1.5-2h).

In [1]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
from src.data.loader import load_config
from src.training.experiments import ExperimentSpec, run_experiment_suite

print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")
cfg = load_config("../config/config.yaml")

CUDA: True | NVIDIA GeForce RTX 5070


In [2]:
# Define the three T5 experiments.
# All share force_fp32=True and optim="adafactor" via extra_trainer_kwargs
# because T5 is numerically unstable in bf16 (see V2 analysis).

t5_specs = [
    ExperimentSpec(
        name="v3_t5_A",
        model_key="t5",
        train_subset=10000,
        num_epochs=2,
        learning_rate=3e-5,
        per_device_batch_size=8,
        gradient_accumulation_steps=2,
        extra_trainer_kwargs={
            "force_fp32": True,
            "optim": "adamw_torch",
            "enable_gradient_checkpointing": False,
        },
    ),
    ExperimentSpec(
        name="v3_t5_B",
        model_key="t5",
        train_subset=10000,
        num_epochs=3,
        learning_rate=1e-4,
        per_device_batch_size=8,
        gradient_accumulation_steps=2,
        extra_trainer_kwargs={
            "force_fp32": True,
            "optim": "adamw_torch",
            "enable_gradient_checkpointing": False,
        },
    ),
    ExperimentSpec(
        name="v3_t5_C",
        model_key="t5",
        train_subset=20000,
        num_epochs=2,
        learning_rate=3e-5,
        per_device_batch_size=8,
        gradient_accumulation_steps=2,
        extra_trainer_kwargs={
            "force_fp32": True,
            "optim": "adamw_torch",
            "enable_gradient_checkpointing": False,
        },
    ),
]

for s in t5_specs:
    print(f"  {s.name}: {s.train_subset} samples, {s.num_epochs} epochs, lr={s.learning_rate}")

  v3_t5_A: 10000 samples, 2 epochs, lr=3e-05
  v3_t5_B: 10000 samples, 3 epochs, lr=0.0001
  v3_t5_C: 20000 samples, 2 epochs, lr=3e-05


In [ ]:
# Run the full suite. Each failed experiment is logged and skipped.
# Total expected time: ~3-4h on RTX 5070.
df_t5 = run_experiment_suite(
    cfg=cfg,
    specs=t5_specs,
    combined_csv_name="v3_t5_search.csv",
    test_subset_size=200,
)
df_t5


[v3_t5_A] Starting experiment
Spec: {'name': 'v3_t5_A', 'model_key': 't5', 'train_subset': 10000, 'num_epochs': 2, 'learning_rate': 3e-05, 'per_device_batch_size': 8, 'gradient_accumulation_steps': 2, 'train_max_input': None, 'lora_overrides': {}, 'extra_trainer_kwargs': {'force_fp32': True, 'optim': 'adamw_torch', 'enable_gradient_checkpointing': False}}


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

{'loss': '4.879', 'grad_norm': '1.987', 'learning_rate': '1.176e-05', 'epoch': '0.08'}
{'loss': '4.295', 'grad_norm': '1.877', 'learning_rate': '2.376e-05', 'epoch': '0.16'}
{'loss': '4.137', 'grad_norm': '1.7', 'learning_rate': '2.936e-05', 'epoch': '0.24'}
{'loss': '4.147', 'grad_norm': '1.694', 'learning_rate': '2.803e-05', 'epoch': '0.32'}
{'loss': '4.061', 'grad_norm': '1.574', 'learning_rate': '2.669e-05', 'epoch': '0.4'}
{'loss': '4.1', 'grad_norm': '1.785', 'learning_rate': '2.536e-05', 'epoch': '0.48'}
{'loss': '4.035', 'grad_norm': '1.715', 'learning_rate': '2.403e-05', 'epoch': '0.56'}
{'loss': '3.991', 'grad_norm': '1.591', 'learning_rate': '2.269e-05', 'epoch': '0.64'}
{'loss': '3.984', 'grad_norm': '1.469', 'learning_rate': '2.136e-05', 'epoch': '0.72'}
{'loss': '4', 'grad_norm': '1.509', 'learning_rate': '2.003e-05', 'epoch': '0.8'}
{'loss': '4.029', 'grad_norm': '1.706', 'learning_rate': '1.869e-05', 'epoch': '0.88'}
{'loss': '4.028', 'grad_norm': '1.402', 'learning_rat

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '4.015', 'grad_norm': '1.456', 'learning_rate': '1.603e-05', 'epoch': '1.04'}
{'loss': '3.958', 'grad_norm': '1.345', 'learning_rate': '1.469e-05', 'epoch': '1.12'}
{'loss': '3.948', 'grad_norm': '1.557', 'learning_rate': '1.336e-05', 'epoch': '1.2'}
{'loss': '3.887', 'grad_norm': '1.481', 'learning_rate': '1.203e-05', 'epoch': '1.28'}
{'loss': '3.914', 'grad_norm': '1.381', 'learning_rate': '1.069e-05', 'epoch': '1.36'}
{'loss': '3.871', 'grad_norm': '1.271', 'learning_rate': '9.36e-06', 'epoch': '1.44'}
{'loss': '3.892', 'grad_norm': '1.321', 'learning_rate': '8.027e-06', 'epoch': '1.52'}
{'loss': '4', 'grad_norm': '1.51', 'learning_rate': '6.693e-06', 'epoch': '1.6'}
{'loss': '3.996', 'grad_norm': '1.809', 'learning_rate': '5.36e-06', 'epoch': '1.68'}
{'loss': '3.952', 'grad_norm': '1.529', 'learning_rate': '4.027e-06', 'epoch': '1.76'}
{'loss': '3.885', 'grad_norm': '1.414', 'learning_rate': '2.693e-06', 'epoch': '1.84'}
{'loss': '3.827', 'grad_norm': '1.487', 'learning_ra

...

{'loss': '4.722', 'grad_norm': '2.052', 'learning_rate': '2.606e-05', 'epoch': '0.08'}
{'loss': '4.174', 'grad_norm': '1.839', 'learning_rate': '5.266e-05', 'epoch': '0.16'}
{'loss': '4.094', 'grad_norm': '1.645', 'learning_rate': '7.926e-05', 'epoch': '0.24'}
{'loss': '4.111', 'grad_norm': '1.728', 'learning_rate': '9.935e-05', 'epoch': '0.32'}
{'loss': '4.028', 'grad_norm': '1.41', 'learning_rate': '9.638e-05', 'epoch': '0.4'}
{'loss': '4.066', 'grad_norm': '1.703', 'learning_rate': '9.342e-05', 'epoch': '0.48'}
{'loss': '3.994', 'grad_norm': '1.67', 'learning_rate': '9.046e-05', 'epoch': '0.56'}
{'loss': '3.952', 'grad_norm': '1.597', 'learning_rate': '8.749e-05', 'epoch': '0.64'}
{'loss': '3.938', 'grad_norm': '1.509', 'learning_rate': '8.453e-05', 'epoch': '0.72'}
{'loss': '3.95', 'grad_norm': '1.511', 'learning_rate': '8.156e-05', 'epoch': '0.8'}
{'loss': '3.971', 'grad_norm': '1.772', 'learning_rate': '7.86e-05', 'epoch': '0.88'}
{'loss': '3.963', 'grad_norm': '1.408', 'learning

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '3.886', 'grad_norm': '1.45', 'learning_rate': '7.267e-05', 'epoch': '1.04'}
{'loss': '3.774', 'grad_norm': '1.359', 'learning_rate': '6.971e-05', 'epoch': '1.12'}
{'loss': '3.771', 'grad_norm': '1.754', 'learning_rate': '6.675e-05', 'epoch': '1.2'}
{'loss': '3.705', 'grad_norm': '1.568', 'learning_rate': '6.378e-05', 'epoch': '1.28'}
{'loss': '3.725', 'grad_norm': '1.383', 'learning_rate': '6.082e-05', 'epoch': '1.36'}
{'loss': '3.694', 'grad_norm': '1.26', 'learning_rate': '5.785e-05', 'epoch': '1.44'}
{'loss': '3.706', 'grad_norm': '1.255', 'learning_rate': '5.489e-05', 'epoch': '1.52'}
{'loss': '3.816', 'grad_norm': '1.487', 'learning_rate': '5.193e-05', 'epoch': '1.6'}
{'loss': '3.807', 'grad_norm': '1.811', 'learning_rate': '4.896e-05', 'epoch': '1.68'}
{'loss': '3.757', 'grad_norm': '1.546', 'learning_rate': '4.6e-05', 'epoch': '1.76'}
{'loss': '3.691', 'grad_norm': '1.388', 'learning_rate': '4.303e-05', 'epoch': '1.84'}
{'loss': '3.632', 'grad_norm': '1.467', 'learning

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '3.586', 'grad_norm': '1.449', 'learning_rate': '3.414e-05', 'epoch': '2.08'}
{'loss': '3.59', 'grad_norm': '1.422', 'learning_rate': '3.118e-05', 'epoch': '2.16'}
{'loss': '3.652', 'grad_norm': '1.582', 'learning_rate': '2.822e-05', 'epoch': '2.24'}
{'loss': '3.524', 'grad_norm': '1.575', 'learning_rate': '2.525e-05', 'epoch': '2.32'}
{'loss': '3.607', 'grad_norm': '1.37', 'learning_rate': '2.229e-05', 'epoch': '2.4'}
{'loss': '3.604', 'grad_norm': '1.653', 'learning_rate': '1.932e-05', 'epoch': '2.48'}
{'loss': '3.524', 'grad_norm': '1.474', 'learning_rate': '1.636e-05', 'epoch': '2.56'}
{'loss': '3.589', 'grad_norm': '1.492', 'learning_rate': '1.34e-05', 'epoch': '2.64'}
{'loss': '3.609', 'grad_norm': '1.323', 'learning_rate': '1.043e-05', 'epoch': '2.72'}


In [ ]:
# Visualization: ROUGE-L by configuration
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

if len(df_t5) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=df_t5, x="experiment", y="rougeL", ax=ax, palette="viridis")

    # Annotate bars with values
    for i, v in enumerate(df_t5["rougeL"]):
        ax.text(i, v + 0.2, f"{v:.2f}", ha="center", fontsize=10)

    # V2 baseline reference line
    ax.axhline(y=23.25, color="red", linestyle="--", alpha=0.7, label="V2 baseline (23.25)")

    ax.set_title("V3 — Flan-T5-base hyperparameter search (ROUGE-L)")
    ax.set_ylabel("ROUGE-L")
    ax.set_xlabel("Experiment")
    ax.set_ylim(bottom=min(df_t5["rougeL"].min() - 1, 22))
    ax.legend()

    plt.tight_layout()
    plt.savefig("../results/figures/v3_t5_search.png", bbox_inches="tight", dpi=120)
    plt.show()

In [ ]:
# Identify the winning configuration
if len(df_t5) > 0:
    best = df_t5.iloc[0]  # already sorted by rougeL desc
    print(f"   Best T5 config: {best['experiment']}")
    print(f"   ROUGE-L: {best['rougeL']:.2f}")
    print(f"   ROUGE-1: {best['rouge1']:.2f}")
    print(f"   ROUGE-2: {best['rouge2']:.2f}")
    print(f"   Delta vs V2 (rougeL=23.25): {best['rougeL'] - 23.25:+.2f}")
    print(f"   Training time: {best['train_minutes']:.1f} min")

## Análisis de V3 — Flan-T5-base

*(Esta sección se rellenará con el análisis concreto cuando terminen los tres
experimentos. Puntos a cubrir:)*

1. **Qué configuración ganó y por qué**
2. **Impacto relativo de cada hiperparámetro**:
   - ¿Más datos ayudaron más que más epochs?
   - ¿El learning rate alto mejoró o desestabilizó?
3. **Comparativa contra V2** — ¿superamos los +0.5 puntos que marcarían una
   mejora estadísticamente significativa sobre 200 muestras?
4. **Selección final para V3b** (LLM-as-judge + interpretabilidad)